# CME Futures: Linear Models

This notebook fits the declared linear-model population for both return horizons. Each model
configuration uses the same walk-forward folds and eligibility rules established by
`05_evaluation`. Ridge, Lasso, and ElasticNet differ only in their regularization parameters.

The notebook executes models and publishes complete validation predictions. `13_backtest`
evaluates every configuration and checkpoint with the equal-weight signal baseline. Validation
backtest Sharpe performs selection.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures linear-model population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

The request table is the model population. A canonical run snapshots every expected prediction
identity before fitting, including every declared checkpoint. A preview must name its reductions
and writes to a separate workspace.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog("linear", labels=ALL_LABELS)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,date,date,i64,str,str
"""linear""","""fwd_ret_21d""","""enet_f0.015""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,1,"""canonical""","""f2c5ff96d40a"""
"""linear""","""fwd_ret_21d""","""enet_f0.03""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,1,"""canonical""","""0416a1995dff"""
"""linear""","""fwd_ret_21d""","""enet_f0.08""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,1,"""canonical""","""1c878f0c99fb"""
"""linear""","""fwd_ret_21d""","""enet_f0.2""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,1,"""canonical""","""a524753d5248"""
"""linear""","""fwd_ret_21d""","""enet_f0.35""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,1,"""canonical""","""9b9c139b4dad"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""linear""","""fwd_ret_5d""","""ridge_a1000.0""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,1,"""canonical""","""0898b6bee178"""
"""linear""","""fwd_ret_5d""","""ridge_a10000.0""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,1,"""canonical""","""6cbd76f173ef"""
"""linear""","""fwd_ret_5d""","""ridge_a100000.0""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,1,"""canonical""","""cef0fb04ac50"""


## Execute and validate

The shared linear runner owns preprocessing, fold fitting, fitted-state digests, restart, metric
computation, and exact eligible-key checks. Any missing or partial member fails the cell.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme-linear-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "config_name", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("linear execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""linear""","""fwd_ret_21d""","""enet_f0.015""","""final""",null,"""canonical""",true,"""f2c5ff96d40a""","""133106340e01"""
"""linear""","""fwd_ret_21d""","""enet_f0.03""","""final""",null,"""canonical""",true,"""0416a1995dff""","""e593217bf8ab"""
"""linear""","""fwd_ret_21d""","""enet_f0.08""","""final""",null,"""canonical""",true,"""1c878f0c99fb""","""a066c4f80cd4"""
"""linear""","""fwd_ret_21d""","""enet_f0.2""","""final""",null,"""canonical""",true,"""a524753d5248""","""9498307c8690"""
"""linear""","""fwd_ret_21d""","""enet_f0.35""","""final""",null,"""canonical""",true,"""9b9c139b4dad""","""637fcfd6245b"""
…,…,…,…,…,…,…,…,…
"""linear""","""fwd_ret_5d""","""ridge_a1000.0""","""final""",null,"""canonical""",true,"""0898b6bee178""","""0ec3a865852d"""
"""linear""","""fwd_ret_5d""","""ridge_a10000.0""","""final""",null,"""canonical""",true,"""6cbd76f173ef""","""0dec31d85309"""
"""linear""","""fwd_ret_5d""","""ridge_a100000.0""","""final""",null,"""canonical""",true,"""cef0fb04ac50""","""ad5bc475c886"""
